# NB05 — Synthesis, LaTeX tables & paper numbers *(light, CPU)*

Gathers the outputs from NB03 (discrimination/operation/calibration) and NB04 (safety/cascade) and produces:
- **Table 1 (LaTeX):** discrimination per model (pooled AUROC/AUPRC + 95% CI).
- **Table 2 (LaTeX):** operating points of the recommended model (sens@90/95/99, spec@95, Youden).
- **Table 3 (LaTeX):** safety & cascade by target sensitivity (filtered workload, missed pathology).
- **Table 4 (LaTeX):** miss by pathology (target-sens 95%).
- **`paper_numbers.json`:** all numbers ready to be cited in the text.

**Narrative (honest — limitation is a contribution):** the binary gate reduces reading workload with high NPV,
**but** it misses rare pathologies (neoplasia/ectasia) and **does not** maintain target-sensitivity across both centers
with a single threshold → conclusion: the gate is useful as a **pre-filter with a safeguard**, not as an isolated decider.


In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'configs').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from configs import config as C
from src import utils
import numpy as np, pandas as pd

CONS = utils.load_json(C.METRICS_DIR / 'consolidated.json')
SAFE = utils.load_json(C.METRICS_DIR / 'nb04_gate_safety.json')
best = CONS['best_model_auprc']
print('recommended model:', best)


## LaTeX Helpers

In [ ]:
def esc(s):
    return str(s).replace('_', r'\_')

def latex_table(df, caption, label, float_fmt='%.3f', col_fmt=None):
    cols = list(df.columns)
    col_fmt = col_fmt or ('l' + 'r'*(len(cols)-1))
    head = ' & '.join(esc(c) for c in cols) + r' \\'
    body = []
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, float):
                cells.append('--' if (v != v) else float_fmt % v)
            else:
                cells.append(esc(v))
        body.append(' & '.join(cells) + r' \\')
    return '\n'.join([
        r'\begin{table}[t]', r'\centering', f'\\caption{{{caption}}}', f'\\label{{{label}}}',
        f'\\begin{{tabular}}{{{col_fmt}}}', r'\toprule', head, r'\midrule',
        *body, r'\bottomrule', r'\end{tabular}', r'\end{table}'])

LATEX_DIR = C.METRICS_DIR / 'latex'; LATEX_DIR.mkdir(exist_ok=True)
def save_tex(text, name):
    (LATEX_DIR / name).write_text(text, encoding='utf-8')
    print('saved:', LATEX_DIR / name)


## Table 1 — Discrimination by model (pooled AUROC/AUPRC + 95% CI)

In [ ]:
rows = []
fam = C.MODEL_FAMILY
for r in CONS['discrimination']:
    rows.append({'Model': r['model'], 'Family': fam.get(r['model'], ''),
                 'AUROC': r['auroc_pool'], '95\% CI (ROC)': f"[{r['auroc_lo']:.3f}, {r['auroc_hi']:.3f}]",
                 'AUPRC': r['auprc_pool'], '95\% CI (PR)': f"[{r['auprc_lo']:.3f}, {r['auprc_hi']:.3f}]"})
t1 = pd.DataFrame(rows)
display(t1.round(3))
tex1 = latex_table(t1[['Model','AUROC','95\% CI (ROC)','AUPRC','95\% CI (PR)']],
                   'Discrimination of the binary triage gate (pooled over 5 folds, bootstrap 95\% CI).',
                   'tab:discrimination')
save_tex(tex1, 'tab1_discrimination.tex')


## Table 2 — Operating points of the recommended model

In [ ]:
op = CONS['operating_points'][best]
order = [('youden','Youden J'), ('sens>=90','Sens@90'), ('sens>=95','Sens@95'),
         ('sens>=99','Sens@99'), ('spec>=95','Spec@95')]
rows = []
for k, lab in order:
    d = op[k]
    rows.append({'Operating point': lab, 'Threshold': d['threshold'],
                 'Sensitivity': d['sensitivity'], 'Specificity': d['specificity'],
                 'PPV': d['ppv'], 'NPV': d['npv'], 'F1': d['f1'], 'MCC': d['mcc']})
t2 = pd.DataFrame(rows)
display(t2.round(3))
tex2 = latex_table(t2, f'Operating points of the recommended gate ({esc(best)}).', 'tab:operating')
save_tex(tex2, 'tab2_operating_points.tex')


## Table 3 — Safety & cascade by target sensitivity

In [ ]:
rows = []
for r in SAFE['operation_at_target_sens']:
    rows.append({'Target sens.': f"{int(r['target_sens']*100)}\%",
                 'Threshold': r['threshold'],
                 'Achieved sens.': r['sensitivity'], 'Spec.': r['specificity'], 'NPV': r['npv'],
                 'Workload filtered': f"{r['fraction_filtered']*100:.1f}\%",
                 'Pathology missed': f"{r['pathology_missed']}/{r['total_pathology']}",
                 'Miss rate': f"{r['miss_rate_pathology']*100:.1f}\%"})
t3 = pd.DataFrame(rows)
display(t3)
tex3 = latex_table(t3, 'Gate safety and cascade workload reduction by target sensitivity.', 'tab:safety')
save_tex(tex3, 'tab3_safety_cascade.tex')


## Table 4 — Miss by pathology (target-sens 95%)

In [ ]:
pm = SAFE['per_pathology_miss_at_sens95']
rows = []
for r in pm['rows']:
    rows.append({'Pathology': r['pathology'], 'Cases': r['total'],
                 'Missed': r['missed'], 'Detected': r['detected'],
                 'Miss rate': (r['miss_rate'] if r['miss_rate'] is not None else float('nan'))})
t4 = pd.DataFrame(rows)
display(t4.round(3))
tex4 = latex_table(t4, f"Per-pathology miss at 95\% target sensitivity (threshold={pm['threshold']:.3f}).",
                   'tab:perpath')
save_tex(tex4, 'tab4_per_pathology.tex')


## Numbers ready for the text (`paper_numbers.json`)
Collects the values that go in the body of the article — so you can cite without recalculating.

In [ ]:
best_disc = next(r for r in CONS['discrimination'] if r['model'] == best)
op95 = next(r for r in SAFE['operation_at_target_sens'] if r['target_sens'] == 0.95)
casc95 = next(r for r in SAFE['cascade'] if r['target_sens'] == 0.95)
cen = SAFE['by_center_at_sens95']['rows']
# rare missed pathologies (for the limitation sentence)
rare = [r for r in SAFE['per_pathology_miss_at_sens95']['rows']
        if r['total'] <= 50 and r['missed'] > 0]

paper = {
    'dataset': {'n_effective': 1984, 'note': 'after drop_neither (6 neither-nor removed)'},
    'best_model': best,
    'discrimination': {
        'auroc': round(best_disc['auroc_pool'], 3),
        'auroc_ci': [round(best_disc['auroc_lo'],3), round(best_disc['auroc_hi'],3)],
        'auprc': round(best_disc['auprc_pool'], 3),
        'auprc_ci': [round(best_disc['auprc_lo'],3), round(best_disc['auprc_hi'],3)],
    },
    'calibration_best': CONS['calibration'][best],
    'operation_sens95': {
        'specificity': round(op95['specificity'], 3), 'npv': round(op95['npv'], 3),
        'workload_filtered_pct': round(op95['fraction_filtered']*100, 1),
        'pathology_missed': op95['pathology_missed'], 'total_pathology': op95['total_pathology'],
        'miss_rate_pathology_pct': round(op95['miss_rate_pathology']*100, 1),
    },
    'cascade_sens95': {
        'workload_reduction_pct': round(casc95['workload_reduction_pct'], 1),
        'pathology_missed': casc95['pathology_missed'],
    },
    'limitation_rare_pathologies': [
        {'pathology': r['pathology'], 'missed': r['missed'], 'total': r['total']} for r in rare
    ],
    'by_center_sens95': [
        {'LOCAL': r['LOCAL'], 'sensitivity': round(r['sensitivity'],3),
         'fraction_filtered': round(r['fraction_filtered'],3),
         'pathology_missed': r['pathology_missed'], 'total_pathology': r['total_pathology']}
        for r in cen
    ],
}
utils.save_json(paper, C.METRICS_DIR / 'paper_numbers.json')
print(json.dumps(paper, ensure_ascii=False, indent=2))


## Ready-made sentences (EN) to paste in the article
Generated from the numbers above — review before using.

In [ ]:
d = paper
frases = [
 f"The best classifier (\\textbf{{{esc(best)}}}) achieved AUROC {d['discrimination']['auroc']:.3f} "
 f"(95\% CI {d['discrimination']['auroc_ci']}) and AUPRC {d['discrimination']['auprc']:.3f} "
 f"(95\% CI {d['discrimination']['auprc_ci']}) in 5-fold cross-validation (n={d['dataset']['n_effective']}).",
 f"Operating at a target sensitivity of 95\%, the gate filters {d['operation_sens95']['workload_filtered_pct']}\% "
 f"of the exams (NPV {d['operation_sens95']['npv']:.3f}), avoiding the referral of "
 f"{d['operation_sens95']['pathology_missed']} out of {d['operation_sens95']['total_pathology']} exams with pathology "
 f"({d['operation_sens95']['miss_rate_pathology_pct']}\%).",
 "However, the binary triage does not distinguish rare findings: " +
 ', '.join(f"{esc(r['pathology'])} ({r['missed']}/{r['total']} missed)" for r in d['limitation_rare_pathologies']) +
 ", highlighting that the gate should operate with a safeguard for rare classes.",
 "With a single threshold, sensitivity was not maintained homogeneously across centers (" +
 '; '.join(f"LOCAL {r['LOCAL']}: sens {r['sensitivity']:.2f}" for r in d['by_center_sens95']) +
 "), indicating a need for per-center calibration.",
]
for f in frases:
    print('•', f, '\n')
(C.METRICS_DIR/'paper_phrases_en.txt').write_text('\n\n'.join(frases), encoding='utf-8')
print('saved: results/metrics/paper_phrases_en.txt')
print('\nEND of A4 pipeline. LaTeX tables in results/metrics/latex/, numbers in paper_numbers.json.')
